In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: una Qiskit Function di Q-CTRL Fire Opal
*Vedi il [riferimento API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile esclusivamente per gli utenti del piano IBM Quantum&reg; Premium, del piano Flex e del piano On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Panoramica
Con il Fire Opal Optimization Solver puoi risolvere problemi di ottimizzazione su scala utility su hardware quantistico senza dover possedere competenze specifiche nel campo quantistico. Inserisci semplicemente la definizione del problema ad alto livello e il Solver si occupa del resto. L'intero flusso di lavoro è noise-aware e sfrutta internamente il [Performance Management di Fire Opal](/guides/q-ctrl-performance-management). Il Solver fornisce in modo costante soluzioni accurate a problemi classicamente difficili, anche alla scala completa del dispositivo sui più grandi QPU IBM&reg;.

Il Solver è flessibile e può essere utilizzato per risolvere problemi di ottimizzazione combinatoria definiti come funzioni obiettivo o come grafi arbitrari. I problemi non devono necessariamente essere mappati alla topologia del dispositivo. Sia i problemi non vincolati che quelli vincolati sono risolvibili, a condizione che i vincoli possano essere formulati come termini di penalità. Gli esempi inclusi in questa guida mostrano come risolvere un problema di ottimizzazione su scala utility, sia non vincolato che vincolato, utilizzando diversi tipi di input per il Solver. Il primo esempio riguarda un problema Max-Cut definito su un grafo 3-regolare a 156 nodi, mentre il secondo affronta un problema di Minimum Vertex Cover a 50 nodi definito da una funzione di costo.

Per ottenere l'accesso all'Optimization Solver, [contatta Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrizione della funzione
Il Solver ottimizza e automatizza completamente l'intero algoritmo, dalla soppressione degli errori a livello hardware alla mappatura efficiente del problema e all'ottimizzazione classica a ciclo chiuso. Dietro le quinte, la pipeline del Solver riduce gli errori a ogni fase, abilitando le prestazioni avanzate necessarie per scalare in modo significativo. Il flusso di lavoro sottostante è ispirato al Quantum Approximate Optimization Algorithm (QAOA), un algoritmo ibrido quantistico-classico. Per un riepilogo dettagliato dell'intero flusso di lavoro dell'Optimization Solver, consulta [il manoscritto pubblicato](https://arxiv.org/abs/2406.01743).

![Visualizzazione del flusso di lavoro dell'Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Per risolvere un problema generico con l'Optimization Solver:
1. Definisci il tuo problema come funzione obiettivo, come grafo o come catena di spin `SparsePauliOp`.
2. Collegati alla funzione tramite il Qiskit Functions Catalog.
3. Esegui il problema con il Solver e recupera i risultati.
### Formati di problema accettati
- Rappresentazione in espressione polinomiale di una funzione obiettivo. Idealmente creata in Python con un oggetto SymPy Poly esistente e formattata in stringa utilizzando [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Rappresentazione tramite grafo di un tipo di problema specifico. Il grafo deve essere creato utilizzando la libreria networkx in Python e poi convertito in stringa tramite la funzione networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Rappresentazione in catena di spin di un problema specifico. La catena di spin deve essere rappresentata come oggetto `SparsePauliOp`; consulta la [documentazione](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) per maggiori dettagli.

> **Note:** Se vuoi usare un backend che questa funzione non supporta attualmente, [contatta Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) per aggiungere il supporto.
## Benchmark
I [risultati di benchmarking pubblicati](https://arxiv.org/abs/2406.01743) mostrano che il Solver risolve con successo problemi con oltre 120 qubit, superando persino i risultati precedentemente pubblicati su dispositivi a quantum annealing e ad ioni intrappolati. Le seguenti metriche di benchmark forniscono un'indicazione approssimativa dell'accuratezza e della scalabilità dei tipi di problema sulla base di alcuni esempi. Le metriche effettive possono variare in base a diverse caratteristiche del problema, come il numero di termini nella funzione obiettivo (densità) e la loro località, il numero di variabili e l'ordine polinomiale.

Il "Numero di qubit" indicato non è un limite assoluto, ma rappresenta soglie approssimative entro le quali ci si può aspettare un'accuratezza della soluzione estremamente costante. Problemi di dimensioni maggiori sono stati risolti con successo e si incoraggia la sperimentazione oltre questi limiti.

La connettività arbitraria dei qubit è supportata per tutti i tipi di problema.

| Tipo di problema    | Numero di qubit | Esempio | Accuratezza | Tempo totale (s) | Utilizzo runtime (s) | Numero di iterazioni
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Problemi quadratici con connettività sparsa  | 156 | Max-Cut 3-regolare | 100%     | 1764     | 293          | 16 |
| Ottimizzazione binaria di ordine superiore | 156 | Modello di spin-glass di Ising | 100%      | 1461     | 272          | 16 |
| Problemi quadratici con connettività densa | 50 | Max-Cut completamente connesso | 100%      |  1758    | 268  | 12 |
| Problema vincolato con termini di penalità | 50 | Minimum Vertex Cover pesato con densità di archi 8% | 100%      | 1074     | 215 | 10 |
## Inizia a usarlo
Prima di tutto, autenticati usando la tua [chiave API di IBM Quantum](http://quantum.cloud.ibm.com/). Poi seleziona la Qiskit Function come segue. (Questo snippet presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nell'ambiente locale.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

### 1. Definisci il problema
Puoi eseguire un problema Max-Cut definendo un problema su grafo e specificando `problem_type='maxcut'`.

In [3]:
# %pip install networkx numpy

### 1. Define the problem
You can run a max-cut problem by defining a graph problem and specifying `problem_type='maxcut'`.

In [5]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [6]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

### 2. Esegui il problema
Quando si utilizza il metodo di input basato su grafo, è necessario specificare il tipo di problema.

In [7]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [8]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [9]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Check your Qiskit Function workload's [status](/docs/guides/functions-get-started#check-job-status) or return [results](/docs/guides/functions-get-started#retrieve-results) as follows:

In [10]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

ba5fbdb8-9e71-49bd-8320-79dcdb46de6a
QUEUED


### 3. Recupera il risultato
Recupera il valore del taglio ottimale dal dizionario dei risultati.

> **Note:** La mappatura delle variabili alla bitstring potrebbe essere cambiata. Il dizionario di output contiene un sotto-dizionario `variables_to_bitstring_index_map` che aiuta a verificare l'ordinamento.

In [11]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


You can verify the accuracy of the result by solving the problem classically with open-source solvers like [PuLP](https://coin-or.github.io/pulp/) if the graph is not densely connected. High density problems may require advanced classical solvers to validate the solution.

## Example: Constrained optimization
The prior max-cut example is a common quadratic unconstrained binary optimization problem. Q-CTRL's Optimization Solver can be used for various problem types, including constrained optimization. You can solve arbitrary problem types by inputting the problem definition represented as a polynomial where constraints are modeled as penalty terms.

The following example demonstrates how to construct a cost function for a constrained optimization problem, [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the following packages to run this example: `numpy`, `networkx`, and `sympy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [12]:
# %pip install numpy networkx sympy

Puoi verificare l'accuratezza del risultato risolvendo il problema classicamente con solver open-source come [PuLP](https://coin-or.github.io/pulp/) se il grafo non è densamente connesso. I problemi ad alta densità potrebbero richiedere solver classici avanzati per validare la soluzione.
## Esempio: ottimizzazione vincolata
Il precedente esempio Max-Cut è un classico problema di ottimizzazione binaria quadratica non vincolata. L'Optimization Solver di Q-CTRL può essere utilizzato per vari tipi di problemi, inclusa l'ottimizzazione vincolata. Puoi risolvere tipi di problemi arbitrari inserendo la definizione del problema rappresentata come polinomio, dove i vincoli sono modellati come termini di penalità.

Il seguente esempio mostra come costruire una funzione di costo per un problema di ottimizzazione vincolata, il [minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Oltre ai pacchetti `qiskit-ibm-catalog` e `qiskit`, per eseguire questo esempio avrai bisogno anche dei seguenti pacchetti: `numpy`, `networkx` e `sympy`. Puoi installare questi pacchetti decommentando la cella seguente se stai eseguendo questo esempio in un notebook con il kernel IPython.

In [13]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

### 1. Definisci il problema
Definisci un problema MVC casuale generando un grafo con nodi pesati casualmente.

In [14]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

![Output della cella di codice precedente](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Un modello di ottimizzazione standard per il MVC pesato può essere formulato come segue. Innanzitutto, è necessario aggiungere una penalità per ogni caso in cui un arco non sia connesso a un vertice nel sottoinsieme. Quindi, sia $n_i = 1$ se il vertice $i$ è nel cover (cioè nel sottoinsieme) e $n_i = 0$ altrimenti. In secondo luogo, l'obiettivo è minimizzare il numero totale di vertici nel sottoinsieme, che può essere rappresentato dalla seguente funzione:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [15]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

Ora ogni arco nel grafo deve includere almeno un estremo del cover, il che può essere espresso con la disuguaglianza:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Ogni caso in cui un arco non è connesso al vertice del cover deve essere penalizzato. Questo può essere rappresentato nella funzione di costo aggiungendo una penalità della forma $P(1-n_i-n_j+n_i n_j)$ dove $P$ è una costante di penalità positiva. Quindi, un'alternativa non vincolata alla disuguaglianza vincolata per il MVC pesato è:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [16]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. Esegui il problema

In [17]:
print(mvc_job.status())

QUEUED


Controlla lo [stato](/guides/functions#check-job-status) del workload della tua Qiskit Function o recupera i [risultati](/guides/functions#retrieve-results) come segue:

In [18]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


## Get support

For any questions or issues, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com).

## Changelog

- 2026-02-11: We now have support for `ibm_miami`

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Q-CTRL Optimization Solver](https://quantum.cloud.ibm.com/functions?id=q-ctrl-optimization-solver).
- Visit the [API reference](/docs/api/functions/q-ctrl-optimization-solver) for this Qiskit Function.
- Try the [Solve higher-order binary optimization problems with Q-CTRL's Optimization Solver](/docs/tutorials/solve-higher-order-binary-optimization-problems-with-q-ctrls-optimization-solver) tutorial.
- Review [Sachdeva, N., et al. (2024).  Quantum optimization using a 127-qubit gate-model IBM quantum computer can outperform quantum annealers for nontrivial binary optimization problems. arXiv preprint arXiv:2406.01743](https://arxiv.org/abs/2406.01743).
- Review [Loco, D., et al. (2026).  Practical protein-pocket hydration-site prediction for drug discovery on a quantum computer. arXiv preprint arXiv:2512.08390](https://arxiv.org/abs/2512.08390).
- Review the [Mazda](https://q-ctrl.com/case-study/tackling-a-costly-bottleneck-in-automotive-design) case study.
- Review the [Network Rail](https://q-ctrl.com/case-study/accelerating-the-schedule-for-quantum-enhanced-rail) case study.
- Review the [Australian Army](https://q-ctrl.com/case-study/improving-army-logistics-with-quantum-computing) case study.
- Review the [Transport for New South Wales](https://q-ctrl.com/case-study/delivering-quantum-computing-for-faster-commuting) case study.

</Admonition>